In [1]:
import pandas as pd
import numpy as np

# load file
df = pd.read_csv("final_data.csv")

# basic info
print(df.shape)
print(df.head())
print(df.info())

ModuleNotFoundError: No module named 'pandas'

NPS Classification

In [ ]:
# create NPS category
def classify_nps(score):
    if score >= 9:
        return "Promoter"
    elif score >= 7:
        return "Passive"
    else:
        return "Detractor"

df["nps_category"] = df["nps_score"].apply(classify_nps)

# distribution
nps_dist = df["nps_category"].value_counts(normalize=True) * 100
print("NPS Distribution (%)")
print(nps_dist)

NPS Distribution (%)
nps_category
Promoter     57.036831
Detractor    25.883416
Passive      17.079753
Name: proportion, dtype: float64


Overall Metrics

In [ ]:
print("Avg NPS:", df["nps_score"].mean())
print("Avg OSAT:", df["osat_score"].mean())

# key scores
score_cols = [
    "osat_service_score",
    "osat_property_appearance_score",
    "osat_guestroom_score",
    "osat_cleanliness_score",
    "osat_checkin_score",
    "osat_checkout_score"
]

print(df[score_cols].mean().sort_values(ascending=False))

Avg NPS: 7.6215796128104305
Avg OSAT: 7.664930615969347
osat_checkout_score               8.764244
osat_checkin_score                8.510881
osat_service_score                8.273445
osat_property_appearance_score    8.123863
osat_cleanliness_score            8.123169
osat_guestroom_score              7.987319
dtype: float64


Problem Impact Analysis

In [ ]:
# % of people facing problems
problem_rate = df["problem_experienced"].mean() * 100
print("Problem rate:", problem_rate)

# NPS with vs without problems
print(df.groupby("problem_experienced")["nps_score"].mean())

Problem rate: 23.71396975641585
problem_experienced
False    8.764918
True     4.076953
Name: nps_score, dtype: float64


Problem Impact Analysis

In [ ]:
hotel_perf = (
    df.groupby("hotel_name")
    .agg(
        avg_nps=("nps_score", "mean"),
        avg_osat=("osat_score", "mean"),
        responses=("nps_score", "count")
    )
    .sort_values(by="avg_nps", ascending=False)
)

print(hotel_perf.head(10))  # top hotels
print(hotel_perf.tail(10))  # worst hotels

                                                    avg_nps  avg_osat  \
hotel_name                                                              
Travelodge by Wyndham San Antonio Lackland AFB ...     10.0      10.0   
Travelodge by Wyndham Santa Maria                      10.0      10.0   
Days Inn by Wyndham Stoughton WI.                      10.0      10.0   
Wyndham Baku                                           10.0       9.5   
Wingate by Wyndham Zhuhai Gongbei Port                 10.0      10.0   
Travelodge by Wyndham Hudsonville                      10.0      10.0   
Travelodge by Wyndham Hope                             10.0      10.0   
Super 8 by Wyndham Brookhaven                          10.0       9.9   
Super 8 by Wyndham Berlin WI                           10.0       9.0   
Days Inn by Wyndham Xi'an Yanliang                     10.0      10.0   

                                                    responses  
hotel_name                                                 

Location Analysis

In [ ]:
state_perf = df.groupby("property_state_code")["nps_score"].mean().sort_values(ascending=False)
print(state_perf)

property_state_code
JL     10.000000
SX      9.985075
YUC     9.545455
NTH     9.470588
SLP     9.321244
         ...    
BR      6.100000
BJ      5.666667
DL      5.423077
GLS     5.000000
GZ      3.000000
Name: nps_score, Length: 169, dtype: float64


Driver Analysis

In [ ]:
corr = df.corr(numeric_only=True)["nps_score"].sort_values(ascending=False)

print("Top positive drivers:")
print(corr.head(10))

print("Top negative drivers:")
print(corr.tail(10))

Top positive drivers:
nps_score                         1.000000
osat_score                        0.965007
osat_guestroom_score              0.869533
osat_service_score                0.849775
osat_cleanliness_score            0.844705
osat_property_appearance_score    0.843986
how_valued_as_member_score        0.824912
osat_food_beverage_score          0.777205
osat_breakfast_score              0.755269
osat_checkin_score                0.745886
Name: nps_score, dtype: float64
Top negative drivers:
osat_food_beverage_score    0.777205
osat_breakfast_score        0.755269
osat_checkin_score          0.745886
osat_checkout_score         0.715661
osat_internet_score         0.689605
wyn_site_id                 0.039204
number_nights               0.008731
room_rate                   0.007026
_updated                   -0.022443
number_guests              -0.163878
Name: nps_score, dtype: float64


Basic Comment Analysis

In [ ]:
# comments length
df["comment_length"] = df["main_comment"].fillna("").apply(len)

print(df.groupby("nps_category")["comment_length"].mean())

nps_category
Detractor    263.177302
Passive      116.075235
Promoter      76.342264
Name: comment_length, dtype: float64


Missing Data Check

In [ ]:
missing = df.isnull().mean() * 100
print(missing.sort_values(ascending=False))

booking_source                         98.953624
offered_rewards_enrollment_checkin     76.887479
number_guests                          74.111629
osat_food_beverage_score               63.390563
osat_breakfast_score                   50.529478
main_comment                           36.557595
how_valued_as_member_score             35.557724
acknowledged_as_member_upon_checkin    35.375514
osat_internet_score                    27.418863
member_level                           27.302980
checkin_time                           20.718626
checkout_time                          20.717482
osat_guestroom_score                   13.034529
osat_service_score                     11.297049
problem_experienced                    10.031868
trip_type                               9.728438
property_state_code                     9.293877
osat_checkout_score                     7.288800
osat_property_appearance_score          6.756654
osat_cleanliness_score                  6.568725
osat_checkin_score  

In [ ]:
import matplotlib.pyplot as plt

# NPS distribution
df["nps_category"].value_counts().plot(kind="bar")
plt.title("NPS Distribution")
plt.show()

# NPS vs problem
df.groupby("problem_experienced")["nps_score"].mean().plot(kind="bar")
plt.title("NPS vs Problem Experienced")
plt.show()